# 📧 Email Sentiment Analysis — Improved NLP Model
### Target Accuracy: **70–75%**

**Key improvements over baseline:**
- ✅ TF-IDF features extracted from email text (subject + body)
- ✅ Combined text + metadata feature matrix
- ✅ SMOTE for class imbalance
- ✅ Tuned XGBoost & LightGBM ensemble
- ✅ Feature importance analysis

## 1. Install & Import

In [ ]:
!pip install kagglehub xgboost lightgbm imbalanced-learn -q

import kagglehub
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from scipy.sparse import hstack, csr_matrix
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords

plt.style.use('ggplot')
print('✅ All libraries loaded!')

## 2. Load Dataset

In [ ]:
path = kagglehub.dataset_download('suvroo/email-sentiment-analysis-dataset')
print('Path to dataset files:', path)

import os
print('Files available:', os.listdir(path))

In [ ]:
df = pd.read_csv(path + '/Email Analysis Dataset.csv')
print(f'Shape: {df.shape}')
display(df.head())

In [ ]:
# Inspect all columns and data types
print('Columns:', df.columns.tolist())
print()
df.info()

In [ ]:
# Check class distribution
print('Sentiment distribution:')
print(df['Sentiment'].value_counts())

plt.figure(figsize=(6,4))
df['Sentiment'].value_counts().plot(kind='bar', color=['steelblue','tomato','seagreen'])
plt.title('Email Sentiment Distribution')
plt.xticks(rotation=0)
plt.show()

## 3. Identify Text Columns

In [ ]:
# Automatically detect text columns (object columns with long average text)
text_columns_candidates = []
for col in df.select_dtypes(include='object').columns:
    avg_len = df[col].dropna().astype(str).apply(len).mean()
    print(f'  {col}: avg_length={avg_len:.1f}')
    if avg_len > 30:  # likely a text column
        text_columns_candidates.append(col)

print(f'\n📝 Detected text columns: {text_columns_candidates}')

In [ ]:
# ── CONFIGURE THIS based on output above ──────────────────────────────────────
# List ALL columns that contain actual email text (subject line, body, message, etc.)
# Examples: ['Email Body', 'Subject', 'Message', 'Email_Body']
# The cell above will show you exactly which columns have long text.

TEXT_COLS = text_columns_candidates  # Auto-detected — adjust if needed
TARGET_COL = 'Sentiment'             # The column we're predicting

print(f'Using text columns: {TEXT_COLS}')
print(f'Target column: {TARGET_COL}')

## 4. Text Preprocessing & NLP Feature Extraction

In [ ]:
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    """Clean and normalize email text for NLP."""
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' url ', text)   # URLs
    text = re.sub(r'\S+@\S+', ' email ', text)               # Email addresses
    text = re.sub(r'[^a-z\s]', ' ', text)                    # Non-alpha chars
    text = re.sub(r'\s+', ' ', text).strip()                  # Extra spaces
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return ' '.join(tokens)

# Combine all text columns into one column
df['combined_text'] = df[TEXT_COLS].fillna('').astype(str).agg(' '.join, axis=1)
df['cleaned_text']  = df['combined_text'].apply(clean_text)

# Preview
print('Sample cleaned text:')
for i in range(3):
    print(f'  [{i}] {df["cleaned_text"].iloc[i][:120]}')

In [ ]:
# ── Hand-crafted Sentiment Features from text ─────────────────────────────────
positive_words = ['great','good','excellent','happy','thank','appreciate','wonderful',
                  'love','perfect','pleased','awesome','fantastic','enjoy','best','glad']
negative_words = ['bad','sorry','issue','problem','urgent','complaint','unhappy','fail',
                  'error','wrong','delay','trouble','disappointed','unfortunately','poor']
neutral_words  = ['meeting','schedule','update','report','information','please','confirm',
                  'attached','review','regarding','kindly','note','reminder','follow']

def count_word_category(text, word_list):
    return sum(1 for w in text.split() if w in word_list)

df['pos_count']   = df['cleaned_text'].apply(lambda t: count_word_category(t, positive_words))
df['neg_count']   = df['cleaned_text'].apply(lambda t: count_word_category(t, negative_words))
df['neu_count']   = df['cleaned_text'].apply(lambda t: count_word_category(t, neutral_words))
df['text_length'] = df['cleaned_text'].apply(len)
df['word_count']  = df['cleaned_text'].apply(lambda t: len(t.split()))
df['excl_count']  = df['combined_text'].apply(lambda t: t.count('!'))
df['ques_count']  = df['combined_text'].apply(lambda t: t.count('?'))
df['upper_ratio'] = df['combined_text'].apply(
    lambda t: sum(1 for c in t if c.isupper()) / max(len(t), 1)
)

print('✅ Hand-crafted text features created')
df[['pos_count','neg_count','neu_count','text_length','word_count']].describe()

In [ ]:
# ── TF-IDF Vectorization ──────────────────────────────────────────────────────
# Using both unigrams and bigrams for richer features
tfidf = TfidfVectorizer(
    max_features=5000,      # top 5000 terms
    ngram_range=(1, 2),     # unigrams + bigrams
    sublinear_tf=True,      # apply log normalization
    min_df=2,               # ignore very rare terms
    max_df=0.95,            # ignore very common terms
    strip_accents='unicode'
)

tfidf_matrix = tfidf.fit_transform(df['cleaned_text'])
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Sample top features: {tfidf.get_feature_names_out()[:20]}')

## 5. Metadata Feature Engineering

In [ ]:
# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Drop non-feature columns
drop_cols = ['email_id', 'from_name', 'to_name', 'combined_text', 'cleaned_text',
             'sentiment',
             # also drop the raw text columns from metadata (already encoded via TF-IDF)
             ] + [c.lower().replace(' ','_') for c in TEXT_COLS]
drop_cols = [c for c in drop_cols if c in df.columns]

# Date features
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['year']      = df['date'].dt.year
    df['month']     = df['date'].dt.month
    df['day']       = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['hour']      = df['date'].dt.hour if df['date'].dt.hour.nunique() > 1 else 0
    drop_cols.append('date')

# Binary Yes/No columns
for col in df.columns:
    if df[col].dtype == 'object' and df[col].dropna().isin(['Yes','No']).all():
        df[col] = df[col].map({'Yes': 1, 'No': 0})

# Encode target
le_target = LabelEncoder()
y = le_target.fit_transform(df['sentiment'])
print(f'Classes: {le_target.classes_}')

# Encode remaining categoricals
meta_df = df.drop(columns=drop_cols, errors='ignore')
for col in meta_df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    meta_df[col] = le.fit_transform(meta_df[col].astype(str))

# Fill any remaining NaNs
meta_df = meta_df.fillna(0)

print(f'Metadata features shape: {meta_df.shape}')
print('Metadata columns:', meta_df.columns.tolist())

## 6. Combine Text + Metadata Features

In [ ]:
from scipy.sparse import hstack, csr_matrix

# Convert metadata to sparse matrix
meta_sparse = csr_matrix(meta_df.values.astype(float))

# Stack TF-IDF + metadata into one feature matrix
X_combined = hstack([tfidf_matrix, meta_sparse])

print(f'✅ Combined feature matrix shape: {X_combined.shape}')
print(f'   TF-IDF features  : {tfidf_matrix.shape[1]}')
print(f'   Metadata features: {meta_sparse.shape[1]}')

## 7. Train/Test Split & SMOTE Balancing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')
print(f'Class distribution in train: {np.bincount(y_train)}')

# Apply SMOTE to balance classes
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After SMOTE train size: {X_train_sm.shape}')
print(f'Class distribution after SMOTE: {np.bincount(y_train_sm)}')

## 8. Model Training

### Model A — XGBoost (tuned)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.08,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='mlogloss',
    tree_method='hist'   # faster training
)

xgb_model.fit(X_train_sm, y_train_sm)
xgb_pred = xgb_model.predict(X_test)

xgb_acc = accuracy_score(y_test, xgb_pred)
print(f'🎯 XGBoost Accuracy: {xgb_acc:.4f} ({xgb_acc*100:.2f}%)')

### Model B — LightGBM (fast gradient boosting)

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.08,
    max_depth=6,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_samples=10,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1
)

lgbm_model.fit(X_train_sm, y_train_sm)
lgbm_pred = lgbm_model.predict(X_test)

lgbm_acc = accuracy_score(y_test, lgbm_pred)
print(f'🎯 LightGBM Accuracy: {lgbm_acc:.4f} ({lgbm_acc*100:.2f}%)')

### Model C — Random Forest (tuned)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_sm, y_train_sm)
rf_pred = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
print(f'🎯 Random Forest Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%)')

### Model D — Soft Voting Ensemble (best of all)

In [ ]:
# Soft voting: combine probability outputs from all 3 models
xgb_proba  = xgb_model.predict_proba(X_test)
lgbm_proba = lgbm_model.predict_proba(X_test)
rf_proba   = rf_model.predict_proba(X_test)

# Weighted average (XGBoost & LightGBM weighted more if they score higher)
ensemble_proba = (0.40 * xgb_proba) + (0.40 * lgbm_proba) + (0.20 * rf_proba)
ensemble_pred  = np.argmax(ensemble_proba, axis=1)

ens_acc = accuracy_score(y_test, ensemble_pred)
print(f'🏆 Ensemble Accuracy: {ens_acc:.4f} ({ens_acc*100:.2f}%)')

## 9. Model Evaluation & Comparison

In [ ]:
# ── Accuracy Comparison Bar Chart ────────────────────────────────────────────
models = ['Baseline\n(metadata only)', 'XGBoost\n(NLP)', 'LightGBM\n(NLP)',
          'Random Forest\n(NLP)', 'Ensemble\n(NLP)']
accs   = [0.48, xgb_acc, lgbm_acc, rf_acc, ens_acc]
colors = ['#aaaaaa', '#4c72b0', '#55a868', '#c44e52', '#8172b2']

plt.figure(figsize=(10, 5))
bars = plt.bar(models, accs, color=colors, edgecolor='black', linewidth=0.8)
plt.axhline(y=0.70, color='red', linestyle='--', linewidth=1.5, label='70% target')
plt.axhline(y=0.75, color='darkred', linestyle='--', linewidth=1.5, label='75% target')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc*100:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.ylim(0, 1.0)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison (Baseline vs NLP-enhanced)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Detailed Classification Report (Best Model) ───────────────────────────────
# Pick the best performing model
best_name = max(
    [('XGBoost', xgb_acc, xgb_pred),
     ('LightGBM', lgbm_acc, lgbm_pred),
     ('RandomForest', rf_acc, rf_pred),
     ('Ensemble', ens_acc, ensemble_pred)],
    key=lambda x: x[1]
)
print(f'\n🏆 Best Model: {best_name[0]} — Accuracy: {best_name[1]*100:.2f}%\n')
print(classification_report(y_test, best_name[2], target_names=le_target.classes_))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, best_name[2])

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_)
plt.title(f'Confusion Matrix — {best_name[0]} ({best_name[1]*100:.1f}%)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## 10. Top TF-IDF Features by Sentiment Class

In [ ]:
# Show most informative words per sentiment class using LightGBM feature importance
# (only works on dense models — let's use TF-IDF top words per class manually)

feature_names = np.array(tfidf.get_feature_names_out())
n_classes = len(le_target.classes_)

fig, axes = plt.subplots(1, n_classes, figsize=(6 * n_classes, 5))
if n_classes == 1:
    axes = [axes]

for i, (cls, ax) in enumerate(zip(le_target.classes_, axes)):
    # Get mean TF-IDF score for documents in this class
    class_mask = (y == i)
    class_tfidf = tfidf_matrix[class_mask].toarray()
    mean_scores = class_tfidf.mean(axis=0)
    top_idx = mean_scores.argsort()[-15:][::-1]
    top_words = feature_names[top_idx]
    top_scores = mean_scores[top_idx]

    ax.barh(top_words[::-1], top_scores[::-1], color='steelblue', edgecolor='black', linewidth=0.5)
    ax.set_title(f'Top Words — "{cls}"', fontweight='bold')
    ax.set_xlabel('Mean TF-IDF Score')

plt.suptitle('Most Informative Words per Sentiment Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Summary & Conclusion

In [ ]:
print('=' * 55)
print('         EMAIL SENTIMENT ANALYSIS — SUMMARY')
print('=' * 55)
print(f'  Baseline (metadata only, no NLP):   48.00%')
print(f'  XGBoost  (TF-IDF + metadata):       {xgb_acc*100:.2f}%')
print(f'  LightGBM (TF-IDF + metadata):       {lgbm_acc*100:.2f}%')
print(f'  Random Forest (TF-IDF + metadata):  {rf_acc*100:.2f}%')
print(f'  Ensemble (weighted soft vote):       {ens_acc*100:.2f}%')
print('=' * 55)
print()
best_acc = max(xgb_acc, lgbm_acc, rf_acc, ens_acc)
improvement = best_acc - 0.48
print(f'  Improvement over baseline: +{improvement*100:.1f} percentage points')
if best_acc >= 0.70:
    print(f'  ✅ TARGET REACHED: {best_acc*100:.1f}% >= 70%')
else:
    print(f'  ⚠️  Best accuracy: {best_acc*100:.1f}% — see tips below')
print('=' * 55)
print()
print('KEY IMPROVEMENTS MADE:')
print('  1. TF-IDF on email text (unigrams + bigrams, 5000 features)')
print('  2. Hand-crafted sentiment indicators (pos/neg/neutral word counts)')
print('  3. Text stats (length, word count, ! ? counts, uppercase ratio)')
print('  4. SMOTE to fix class imbalance')
print('  5. Tuned XGBoost + LightGBM + RF Ensemble')

## 🔧 Optional: Further Improvements (if needed)

If the model still falls below 70%, try these:

```python
# Option A: More TF-IDF features
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 3))

# Option B: Use pre-trained BERT embeddings (best for NLP)
# !pip install sentence-transformers
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer('all-MiniLM-L6-v2')
# embeddings = model.encode(df['cleaned_text'].tolist())

# Option C: Hyperparameter tuning with RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.05, 0.08, 0.1],
    'max_depth': [4, 6, 8],
    'subsample': [0.7, 0.8, 0.9]
}
search = RandomizedSearchCV(XGBClassifier(), param_grid, n_iter=20,
                            cv=3, scoring='accuracy', random_state=42)
search.fit(X_train_sm, y_train_sm)
print('Best params:', search.best_params_)
```